In [1]:
!pip install --upgrade --quiet google-cloud-vectorsearch pandas requests google-adk

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastmcp 2.13.3 requires mcp!=1.21.1,<1.23,>=1.19.0, but you have mcp 1.26.0 which is incompatible.
streamlit 1.50.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.1 which is incompatible.


In [3]:
from google.cloud import bigquery

client = bigquery.Client()


In [ ]:
import os
from google.cloud import vectorsearch_v1beta


PROJECT_ID = "project-843a9db4-70a9-4565-b6c"
LOCATION = "us-central1"
COLLECTION_ID = "london-travel-agent-demo"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

admin_client = vectorsearch_v1beta.VectorSearchServiceClient()
data_client = vectorsearch_v1beta.DataObjectServiceClient()
search_client = vectorsearch_v1beta.DataObjectSearchServiceClient()

parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"
collection_path = f"{parent}/collections/{COLLECTION_ID}"

print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Collection: {COLLECTION_ID}")

Project: project-843a9db4-70a9-4565-b6c
Location: us-central1
Collection: london-travel-agent-demo


In [12]:
import io
import pandas as pd
import requests
import warnings
warnings.filterwarnings('ignore')


# Source: Inside Airbnb (London, Sept 2025)
DATA_URL = "https://data.insideairbnb.com/united-kingdom/england/london/2025-09-14/data/listings.csv.gz"

print('Downloading data...')

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}
response= requests.get(DATA_URL, headers= headers)
response.raise_for_status()

df= pd.read_csv(io.BytesIO(response.content), compression="gzip")
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.87,4.78,4.78,NaN,f,2,1,1,0,0.30
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.84,4.93,4.74,NaN,f,1,1,0,0,0.51
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.72,4.89,4.61,NaN,f,2,2,0,0,0.32
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,4.93,4.60,4.65,NaN,f,1,1,0,0,0.53
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,4.46,4.85,4.54,NaN,t,2,2,0,0,0.09


# Data Cleaning


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96871 entries, 0 to 96870
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            96871 non-null  int64  
 1   listing_url                                   96871 non-null  str    
 2   scrape_id                                     96871 non-null  int64  
 3   last_scraped                                  96871 non-null  str    
 4   source                                        96871 non-null  str    
 5   name                                          96871 non-null  str    
 6   description                                   94421 non-null  str    
 7   neighborhood_overview                         41208 non-null  str    
 8   picture_url                                   96865 non-null  str    
 9   host_id                                       96871 non-null  int64  
 1

In [14]:
df['price']

0         $70.00
1        $149.00
2        $411.00
3            NaN
4        $210.00
          ...   
96866    $298.00
96867     $66.00
96868    $350.00
96869     $40.00
96870    $139.00
Name: price, Length: 96871, dtype: str

In [16]:
cols = [
    "id",
    "name",
    "description",
    "price",
    "neighborhood_overview",
    "listing_url",
    "instant_bookable",
    "neighbourhood_cleansed",
]
df = df[cols].copy()

# clean price column remove $ and convert to float
df['price']= df['price'].astype(str).str.replace("[\$,]", "", regex= True).astype(float)


# Fill NaNs in text fields (Critical to avoid API errors)
str_cols = [
    "name",
    "neighbourhood_cleansed",
    "instant_bookable",
    "listing_url",
    "description",
    "neighborhood_overview",
]
for col in str_cols:
    df[col] = df[col].fillna("").astype(str)

# Normalize boolean string
df["instant_bookable"] = df["instant_bookable"].str.lower()

# Subset for demo speed
df_demo = df.head(2000).reset_index(drop=True)
print(f"Loaded & Cleaned {len(df_demo)} listings.")






Loaded & Cleaned 2000 listings.


In [37]:
df_demo = df_demo.fillna("")

# Ensure numeric price
df_demo["price"] = pd.to_numeric(df_demo["price"], errors="coerce").fillna(0)

# Convert everything to correct type
df_demo["name"] = df_demo["name"].astype(str)
df_demo["neighbourhood_cleansed"] = df_demo["neighbourhood_cleansed"].astype(str)
df_demo["instant_bookable"] = df_demo["instant_bookable"].astype(str)
df_demo["listing_url"] = df_demo["listing_url"].astype(str)
df_demo["description"] = df_demo["description"].astype(str)
df_demo["neighborhood_overview"] = df_demo["neighborhood_overview"].astype(str)


# Part 3: Create Collection
## Create a Vector Search 2.0 Collection
A Collection is a schema-enforced container for your data in Vector Search 2.0. Think of it as a table in a traditional database, but optimized for vector operations.

## Collection Schemas
Each Collection has two schemas:

*Data Schema*: Defines the structure of your data fields using JSON Schema format. All Data Objects must conform to this schema.

*Vector Schema*: Defines your embedding fields with their dimensions and configurations. You can have multiple vector fields per object (e.g., text_embedding, image_embedding).

## Auto-Embeddings Feature
One of Vector Search 2.0's most powerful features is automatic embedding generation. When you configure vertex_embedding_config in your vector schema, the service automatically generates embeddings using Vertex AI models. This means you don't need to:

Manage embedding model infrastructure
Pre-compute embeddings before ingestion
Handle embedding API calls yourself

In [38]:

collection_config = {
    # DATA SCHEMA: Defines the structure of your data fields
    # All fields in Data Objects must match these types
    "data_schema": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},  # Listing title
            "price": {"type": "number"},  # Price per night (GBP)
            "neighborhood": {"type": "string"},  # London neighborhood
            "listing_url": {"type": "string"},  # Airbnb URL
            "instant_bookable": {"type": "string"},  # "t" or "f"
            "description": {"type": "string"},  # Full listing description
            "neighborhood_overview": {"type": "string"},  # Area description
        },
    },
    # VECTOR SCHEMA: Defines embedding fields and their configurations
    "vector_schema": {
        "description_embedding": {
            "dense_vector": {
                # Embedding dimensions (768 for gemini-embedding-001)
                "dimensions": 768,
                # AUTO-EMBEDDING CONFIGURATION
                # Vector Search 2.0 will automatically generate embeddings
                # using the specified Vertex AI model
                "vertex_embedding_config": {
                    "model_id": "gemini-embedding-001",
                    # text_template: Combines multiple fields into embedding input
                    # This creates richer semantic embeddings by including both
                    # the description AND neighborhood context
                    "text_template": "Description: {description}. Neighborhood: {neighborhood_overview}.",
                    # task_type: Optimizes embeddings for retrieval use cases
                    # Use RETRIEVAL_DOCUMENT for documents being indexed
                    "task_type": "RETRIEVAL_DOCUMENT",
                },
            }
        }
    },
}

# Create the Collection (or skip if it already exists)
try:
    existing = admin_client.get_collection(name=collection_path)
    print(f"Collection '{COLLECTION_ID}' already exists.")
except Exception:
    print(f"Creating Collection '{COLLECTION_ID}'...")
    request = vectorsearch_v1beta.CreateCollectionRequest(
        parent=parent, collection_id=COLLECTION_ID, collection=collection_config
    )
    operation = admin_client.create_collection(request=request)
    operation.result()  # Wait for completion
    print(f"Collection '{COLLECTION_ID}' created successfully!")

Collection 'london-travel-agent-demo' already exists.


# Ingest Data object


In [39]:
import time

from tqdm.auto import tqdm

print(f"Ingesting {len(df_demo)} listings into '{COLLECTION_ID}'...")

# Batch size: Max 250 for gemini-embedding-001 auto-embeddings
BATCH_SIZE = 100  # Using 100 for safety margin

# Prepare Data Objects
# Each object needs: data_object_id + data (matching schema) + vectors (empty for auto-embedding)
data_objects = []
for _, row in df_demo.iterrows():
    data_objects.append(
        {
            "data_object_id": str(row["id"]),  # Unique ID (must be string)
            "data_object": {
                "data": {
                    "name": row["name"],
                    "price": float(row["price"]),  # Ensure numeric type
                    "neighborhood": row["neighbourhood_cleansed"],
                    "instant_bookable": row["instant_bookable"],
                    "listing_url": row["listing_url"],
                    "description": row["description"],
                    "neighborhood_overview": row["neighborhood_overview"],
                },
                # Empty vectors = trigger auto-embedding generation
                # Vector Search 2.0 will use the vertex_embedding_config from our schema
                "vectors": {},
            },
        }
    )

# Batch Upload with Progress Bar
for i in tqdm(
    range(0, len(data_objects), BATCH_SIZE), desc="Uploading batches"
):
    batch = data_objects[i : i + BATCH_SIZE]

    try:
        request = vectorsearch_v1beta.BatchCreateDataObjectsRequest(
            parent=collection_path, requests=batch
        )
        data_client.batch_create_data_objects(request)

        # Rate limiting: Pause between batches to respect embedding API quotas
        time.sleep(2)

    except Exception as e:
        # Skip "already exists" errors (useful for re-runs)
        if "already exists" not in str(e).lower():
            tqdm.write(f"Batch error: {str(e)[:80]}")

print(f"\nIngestion complete! {len(data_objects)} listings loaded.")

Ingesting 2000 listings into 'london-travel-agent-demo'...


Uploading batches: 100%|██████████| 20/20 [01:27<00:00,  4.37s/it]


Ingestion complete! 2000 listings loaded.


# vector search tool


In [40]:
import json
from typing import Any

from google.cloud import vectorsearch_v1beta


def find_rentals(query: str, filter: str = "") -> list[dict[str, Any]]:
    """
    Search for vacation rentals using Hybrid Search (Semantic + Keyword) with metadata filtering.

    This function demonstrates Vector Search 2.0's hybrid search capability, combining:
    1. Semantic Search: Understands query intent (e.g., "cozy" finds warm, inviting spaces)
    2. Text Search: Matches exact keywords (e.g., "garden" finds listings with gardens)
    3. RRF Ranking: Merges results using Reciprocal Rank Fusion for balanced relevance

    Args:
        query: Natural language description of desired rental (e.g., "artist loft with garden")
        filter: JSON string with metadata filters (e.g., '{"price": {"$lt": 200}}')

    Returns:
        List of matching rentals with name, price, neighborhood, and URL
    """
    print("\n>>> TOOL CALL: find_rentals (Hybrid Search)")
    print(f"    Query: {query}")
    print(f"    Filter: {filter if filter else 'None'}")

    # Parse Filter JSON (if provided)
    filter_dict = None
    if filter.strip():
        try:
            filter_dict = json.loads(filter)
        except json.JSONDecodeError:
            print("    Warning: Invalid JSON filter, ignoring.")

    try:
        # Configure Semantic Search
        # Uses auto-generated embeddings with QUESTION_ANSWERING task type
        # (pairs with RETRIEVAL_DOCUMENT used during indexing)
        semantic_search = vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field="description_embedding",  # The vector field to search
            filter=filter_dict,  # Metadata filtering supported
            task_type="QUESTION_ANSWERING",  # Optimized for query-document matching
            top_k=10,
            output_fields=vectorsearch_v1beta.OutputFields(
                data_fields=["name", "price", "neighborhood", "listing_url"]
            ),
        )

        # Configure Text Search (Keyword Matching)
        # Searches across multiple text fields for exact keyword matches
        text_search = vectorsearch_v1beta.TextSearch(
            search_text=query,
            data_field_names=["name", "description", "neighborhood_overview"],
            filter=filter_dict,  # Metadata filtering supported
            top_k=10,
            output_fields=vectorsearch_v1beta.OutputFields(
                data_fields=["name", "price", "neighborhood", "listing_url"]
            ),
        )

        # Execute Hybrid Search with RRF Ranking
        # BatchSearchDataObjectsRequest combines multiple searches
        # RRF (Reciprocal Rank Fusion) merges results based on position in each list
        # weights=[0.6, 0.4] gives slightly more importance to semantic search
        request = vectorsearch_v1beta.BatchSearchDataObjectsRequest(
            parent=collection_path,
            searches=[
                vectorsearch_v1beta.Search(semantic_search=semantic_search),
                vectorsearch_v1beta.Search(text_search=text_search),
            ],
            combine=vectorsearch_v1beta.BatchSearchDataObjectsRequest.CombineResultsOptions(
                ranker=vectorsearch_v1beta.Ranker(
                    rrf=vectorsearch_v1beta.ReciprocalRankFusion(
                        weights=[0.6, 0.4]
                    )
                )
            ),
        )

        response = search_client.batch_search_data_objects(request=request)

        # Format Results
        results = []
        if response.results and response.results[0].results:
            for res in response.results[0].results:
                data = res.data_object.data
                results.append(
                    {
                        "name": data.get("name"),
                        "price": data.get("price"),
                        "neighborhood": data.get("neighborhood"),
                        "url": data.get("listing_url"),
                    }
                )

        print(f"    Found: {len(results)} listings")
        return results

    except Exception as e:
        print(f"    Error: {e}")
        return []

# Build the ADK Agent

In [41]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# Agent Instructions
# These teach the agent how to use the find_rentals tool effectively
AGENT_INSTRUCTION = """
You are an expert London Travel Agent helping users find vacation rentals.

You have access to a tool called `find_rentals` with two arguments:
1. `query`: A description of the vibe/place (e.g., "artist loft", "garden flat", "cozy workspace")
2. `filter`: A JSON string to filter results by metadata

### AVAILABLE FILTER FIELDS
- `price` (number): Price per night in GBP
- `neighborhood` (string): London neighborhood (e.g., "Hackney", "Islington", "Camden")
- `instant_bookable` (string): "t" for instantly bookable, "f" for requires approval

### FILTER SYNTAX EXAMPLES
```
Price under £200:           {"price": {"$lt": 200.0}}
Specific neighborhood:      {"neighborhood": {"$eq": "Hackney"}}
Price range:                {"$and": [{"price": {"$gte": 100}}, {"price": {"$lte": 300}}]}
Hackney + Instant Book:     {"$and": [{"neighborhood": {"$eq": "Hackney"}}, {"instant_bookable": {"$eq": "t"}}]}
Complex filter:             {"$and": [{"neighborhood": {"$eq": "Hackney"}}, {"price": {"$lt": 200}}, {"instant_bookable": {"$eq": "t"}}]}
```

### GUIDELINES
- Extract the semantic/vibe part of the request for the `query` parameter
- Extract price, location, and booking constraints for the `filter` parameter
- Always summarize the results in a friendly, helpful manner
- Include prices and URLs when available
"""


travel_agent = Agent(
    name="London_Travel_Agent",
    model= 'gemini-2.5-flash',
    tools=[find_rentals],
    instruction=AGENT_INSTRUCTION,)

runner = InMemoryRunner(agent=travel_agent)

print("Travel agent intiated and Ready")

Travel agent intiated and Ready


# Testing

In [42]:
await runner.run_debug("I want an inspiring workspace in Islington")


 ### Created new session: debug_session_id

User > I want an inspiring workspace in Islington

>>> TOOL CALL: find_rentals (Hybrid Search)
    Query: inspiring workspace
    Filter: {"neighborhood": {"$eq": "Islington"}}
    Found: 10 listings
London_Travel_Agent > Certainly! I found some inspiring workspace options for you in Islington:

*   **Stretch out on the Corner Sofa at a Plant-Filled Getaway** - This sounds like a lovely spot! It's £150 per night. You can find out more here: https://www.airbnb.com/rooms/511429
*   **Ensuite room in artist lofty studio** - Perfect for creative inspiration at £120 per night. Check it out: https://www.airbnb.com/rooms/442178
*   **Architect designed studio flat** - A sleek and modern option for £180 per night: https://www.airbnb.com/rooms/1344695
*   **Double Room and Workspace in spacious Garden Flat** - Enjoy a workspace with a garden for £54 per night: https://www.airbnb.com/rooms/435600
*   **Private room in London & cultural Islington** - A

[Event(model_version='gemini-2.5-flash', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'filter': '{"neighborhood": {"$eq": "Islington"}}',
           'query': 'inspiring workspace'
         },
         id='adk-9a958b94-0629-4760-b520-ccbf4db40268',
         name='find_rentals'
       ),
       thought_signature=b'\n\x94\x03\x01\x8f=k_\x16\xa4\xfdV\x01h{vZ\xady\x92Wq\xc9=~`\xe9\xff\x02/\xd0\xea\x19\xd7\xff\xc1^\x85\xc3\xa5\xb5[\xf5Cl\x8fZ\x11\x9e\xaf\x7f\xef\xd0/\xb2\xa6\xfb\xe8\xcb\x9b\x80S\xee4#\xf1!\xe7\xb8\x90z\x80\xce\xf3\x0c\xbe!x*\xeb\xbf\xd5\xbd3\xec-qi\xb9se\xe8\xfcT\x9d\x01E...'
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=19,
   candidates_tokens_details=[
     ModalityTokenCount

In [43]:
await runner.run_debug(
    "Find me a place in Hackney under £200 that I can book instantly. I want a creative artist vibe."
)


 ### Continue session: debug_session_id

User > Find me a place in Hackney under £200 that I can book instantly. I want a creative artist vibe.

>>> TOOL CALL: find_rentals (Hybrid Search)
    Query: creative artist vibe
    Filter: {"$and": [{"neighborhood": {"$eq": "Hackney"}}, {"price": {"$lt": 200}}, {"instant_bookable": {"$eq": "t"}}]}
    Found: 10 listings
London_Travel_Agent > I've found a few places in Hackney with a creative artist vibe that are under £200 and available for instant booking!

Here are some options:

*   **The Residential Suite Above Gallery** - This sounds perfect for an artist! It's £131 per night. Check it out here: https://www.airbnb.com/rooms/173082
*   **Characterful Warehouse Apartment in the Heart of Shoreditch** - Get that industrial artist feel for £198 per night: https://www.airbnb.com/rooms/3008678
*   **Shoreditch Loft** - Another great loft option for £151 per night: https://www.airbnb.com/rooms/227502
*   **Hackney Stylish & light 1 bedroom Vict

[Event(model_version='gemini-2.5-flash', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'filter': '{"$and": [{"neighborhood": {"$eq": "Hackney"}}, {"price": {"$lt": 200}}, {"instant_bookable": {"$eq": "t"}}]}',
           'query': 'creative artist vibe'
         },
         id='adk-aea87697-eb17-4945-aa5b-275f1b2e64af',
         name='find_rentals'
       ),
       thought_signature=b"\n\xd3\x04\x01\x8f=k_\xe5?)o\x9b\x95\x06\xa7\x8c(\x11\xc1\xb3\xd4'\xca:y\xd4\xb8p\xa1\x9d\x04\xa6\xbe\x00\xa8%\xb2u\xb79S\xd8)Y\xed\xc7Z.e\x15\xd7\xa6\xf8\x1b\xf0\xe3\x19u\xae\xcdV\xdb\x01\xe1^h\x8cc\x9f\x1f\xb0c\xaf\x94\xea\x1e8\xeb\x87\xf1L{\xe9!7u\xd2x$\x15%k\xde%\xf5H...'
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_t

# Cleanup Resources

In [45]:

DELETE_COLLECTION = True

if DELETE_COLLECTION:
    try:
        print(f"Deleting all Data Objects from '{COLLECTION_ID}'...")

        # Delete all Data Objects first (required before Collection deletion)
        deleted_count = 0
        while True:
            query_request = vectorsearch_v1beta.QueryDataObjectsRequest(
                parent=collection_path,
                page_size=100,
                output_fields=vectorsearch_v1beta.OutputFields(data_fields=[]),
            )
            results = list(search_client.query_data_objects(query_request))

            if not results:
                break

            for obj in results:
                try:
                    delete_request = (
                        vectorsearch_v1beta.DeleteDataObjectRequest(
                            name=obj.name
                        )
                    )
                    data_client.delete_data_object(delete_request)
                    deleted_count += 1
                except Exception:
                    pass

            print(f"  Deleted {deleted_count} data objects...")

        print(f"Deleted {deleted_count} total data objects.")

        # Now delete the Collection
        print(f"Deleting Collection '{COLLECTION_ID}'...")
        request = vectorsearch_v1beta.DeleteCollectionRequest(
            name=collection_path
        )
        operation = admin_client.delete_collection(request=request)
        operation.result()
        print(f"Collection '{COLLECTION_ID}' deleted successfully.")

    except Exception as e:
        print(f"Error during cleanup: {e}")
else:
    print("Cleanup skipped. Set DELETE_COLLECTION = True to delete resources.")

Deleting all Data Objects from 'london-travel-agent-demo'...
  Deleted 2000 data objects...
Deleted 2000 total data objects.
Deleting Collection 'london-travel-agent-demo'...
Collection 'london-travel-agent-demo' deleted successfully.
